In [ ]:
import pandas as pd
import os

df = pd.read_csv("/kaggle/input/computer-vision-project-dataset/poster_image_scores.csv")

df['poster_title'] = df['image_path'].apply(lambda x: os.path.splitext(os.path.basename(x))[0].replace('_', ' '))

print(df[['image_path', 'poster_title']].head())

In [ ]:
face_matches_df = pd.read_csv("/kaggle/input/computer-vision-project-dataset/face_matches.csv")

In [ ]:
df[df.image_path == "poster_images/poster_images/_Tis_the_Season_for_Love_2015_.jpg"]

In [ ]:
df.to_csv("tesnime.csv",index=False)

In [ ]:
df.shape

In [ ]:
image_with_actors_df = pd.read_csv("/kaggle/input/computer-vision-project-dataset/poster_image_scores_with_actors.csv")

In [ ]:
image_with_actors_df = image_with_actors_df[["image_path","popularity"]]


In [ ]:
combined_df = pd.merge(
    image_with_actors_df,
    df,
    on='image_path',
    how='right'  
)


print(combined_df.head())
print(f"Combined row count: {len(combined_df)}")

In [ ]:
bert_embeddings = pd.read_csv("/kaggle/input/embeddings-1/titles_with_bert_embeddings_1.csv")

In [ ]:
bert_embeddings['bert_cls_embedding'] = bert_embeddings['bert_cls_embedding'].apply(
    lambda x: [float(i) for i in x.split(',')] if isinstance(x, str) else x
)

print(len(bert_embeddings['bert_cls_embedding'].iloc[0]))
print(type(bert_embeddings['bert_cls_embedding'].iloc[0])) 

In [ ]:
combined_df.shape

In [ ]:
bert_embeddings= bert_embeddings[["image_path","bert_cls_embedding"]]

In [ ]:
final_df = pd.merge(
    combined_df,
    bert_embeddings,
    on='image_path',
    how='right'  
)


print(final_df.head())
print(f"Final row count: {len(final_df)}")

In [ ]:
final_df.rename(columns={'popularity': 'actor_score'}, inplace=True)
final_df.rename(columns={'bert_cls_embedding': 'title_embedding'}, inplace=True)

In [ ]:
import os, random, time, gc
import pandas as pd
from PIL import Image, UnidentifiedImageError
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
import lightning as L
from torchmetrics.regression import MeanAbsoluteError, MeanSquaredError

# Set random seed for reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# Define configurable parameters for training
POSTER_DIR = "/kaggle/input/computer-vision-project-dataset/poster_images/"
IMG_COL    = "image_path"
SCORE_COL  = "imdb_score"
EPOCHS     = 10
BATCH_SIZE = 128
LR         = 1e-4


# Custom dataset class for loading and preprocessing poster images
class PosterDS(Dataset):
    def __init__(self, df, root, tfm):
        self.df = df.reset_index(drop=True)
        self.root = root
        self.tfm = tfm

        # Normalize actor scores to range [0, 1]
        min_score = self.df["actor_score"].min()
        max_score = self.df["actor_score"].max()
        self.df["actor_score_norm"] = (self.df["actor_score"] - min_score) / (max_score - min_score)

    def __len__(self): 
        return len(self.df)

    def __getitem__(self, idx):
        # Load image and preprocess it; handle errors gracefully
        row = self.df.iloc[idx]
        path = os.path.join(self.root, row[IMG_COL])

        try:
            img = Image.open(path).convert("RGB")
        except (FileNotFoundError, UnidentifiedImageError):
            return self.__getitem__((idx + 1) % len(self))

        x_img = self.tfm(img)
        y = torch.tensor(row[SCORE_COL], dtype=torch.float32)
        actor_score = torch.tensor(row["actor_score_norm"], dtype=torch.float32).unsqueeze(0)
        return x_img, actor_score, y


# Define a model combining ViT features and actor scores
class ViTWithFaceOnly(nn.Module):
    def __init__(self):
        super().__init__()
        # Load pretrained ViT model and modify its head
        self.vit = timm.create_model("vit_base_patch16_224", pretrained=True)
        vit_out_dim = self.vit.head.in_features
        self.vit.head = nn.Identity()

        # Projection layer for actor scores
        self.face_proj = nn.Sequential(
            nn.Linear(1, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Final regression head combining ViT and actor score features
        self.head = nn.Sequential(
            nn.Linear(vit_out_dim + 128, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 1)
        )

    def forward(self, x_img, x_actor):
        # Forward pass through ViT and actor score projection
        x_vit = self.vit(x_img)
        x_face = self.face_proj(x_actor)
        x = torch.cat([x_vit, x_face], dim=1)
        return self.head(x).squeeze(1)

# Define a LightningModule for training and validation
class Regr(L.LightningModule):
    def __init__(self, model, lr=LR):
        super().__init__()
        self.model = model
        self.lr = lr
        self.mae = MeanAbsoluteError()
        self.mse = MeanSquaredError()

    def forward(self, x_img, x_actor):
        return self.model(x_img, x_actor)

    def _step(self, batch, tag):
        # Compute loss and log metrics for a batch
        x_img, x_actor, y = batch
        yhat = self(x_img, x_actor)
        loss = nn.MSELoss()(yhat, y)
        self.log(f"{tag}_mae", self.mae(yhat, y), prog_bar=True, batch_size=len(y))
        self.log(f"{tag}_mse", self.mse(yhat, y), prog_bar=True, batch_size=len(y))
        return loss

    def training_step(self, b, i): 
        return self._step(b, "train")
    
    def validation_step(self, b, i): 
        return self._step(b, "val")

    def configure_optimizers(self):
        # Define optimizer and learning rate scheduler
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        return {"optimizer": optimizer, "lr_scheduler": scheduler}

# Function to build training and validation dataloaders
def build_loaders(df, img_size=224, bs=32, workers=2):
    # Filter out rows with missing image files
    df = df[df[IMG_COL].apply(lambda p: os.path.exists(os.path.join(POSTER_DIR, p)))]
    val_df = df.sample(frac=0.1, random_state=SEED)
    tr_df  = df.drop(val_df.index)

    # Define image transformations for training and validation
    mean, std = [0.5]*3, [0.5]*3
    tr_tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(.2, .2, .2, .1),
        transforms.ToTensor(), transforms.Normalize(mean, std)])
    val_tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(), transforms.Normalize(mean, std)])

    # Create dataloaders for training and validation
    tr_loader = DataLoader(
        PosterDS(tr_df, POSTER_DIR, tr_tfm),
        batch_size=bs, shuffle=True, num_workers=workers, pin_memory=True)
    val_loader = DataLoader(
        PosterDS(val_df, POSTER_DIR, val_tfm),
        batch_size=bs, shuffle=False, num_workers=workers, pin_memory=True)
    return tr_loader, val_loader


# Function to train the model and evaluate its performance
def train_one(df, epochs=EPOCHS, bs=BATCH_SIZE, freeze_vit=False):
    # Build dataloaders
    tr_loader, val_loader = build_loaders(df, bs=bs)
    model = ViTWithFaceOnly()

    # Optionally freeze ViT backbone
    if freeze_vit:
        for param in model.vit.parameters():
            param.requires_grad = False
        print("ViT backbone is frozen.")
    else:
        print("ViT backbone is trainable.")

    # Wrap model in LightningModule
    lit_model = Regr(model)

    # Define checkpoint callback
    checkpoint_cb = L.pytorch.callbacks.ModelCheckpoint(
        monitor="val_mae", mode="min", save_top_k=1,
        filename="best-vit-face-{epoch:02d}-{val_mae:.3f}"
    )

    # Print dataset sizes
    print(f"Training on {len(tr_loader.dataset):,} samples")
    print(f"Validating on {len(val_loader.dataset):,} samples")

    # Initialize Lightning trainer
    trainer = L.Trainer(
        max_epochs=epochs,
        precision="16-mixed" if torch.cuda.is_available() else 32,
        accelerator="auto",
        deterministic=True,
        log_every_n_steps=10,
        callbacks=[checkpoint_cb]
    )

    # Train and validate the model
    t0 = time.time()
    trainer.fit(lit_model, tr_loader, val_loader)
    metrics = trainer.validate(lit_model, val_loader, verbose=False)[0]
    print(f"\nFINAL MAE={metrics['val_mae']:.3f}  MSE={metrics['val_mse']:.3f}  |  {(time.time() - t0)/60:.1f} min")
    print(f"Best checkpoint saved to: {checkpoint_cb.best_model_path}")

    # Display sample predictions
    lit_model.eval()
    with torch.no_grad():
        for x_img, x_actor, y_true in val_loader:
            y_pred = lit_model(x_img.to(lit_model.device), x_actor.to(lit_model.device))
            print("\nSample predictions:")
            for i in range(min(5, len(y_true))):
                print(f"True: {y_true[i].item():.2f} → Pred: {y_pred[i].item():.2f}")
            break

    return metrics


# Train the model and print final metrics
metrics = train_one(final_df)

print("\nFinal Metrics:")
print("MAE:", metrics["val_mae"], "MSE:", metrics["val_mse"])